# 01 · Data exploration — NASA C-MAPSS FD001

**Goal:** understand the data before modelling: structure, quality, which sensors carry a degradation signal, how balanced the label is and where leakage could appear.

**Rules for this notebook:** it only *explores*. Reusable logic lives in `src/industrial_ai/` and decisions are recorded in `docs/ml.md`.

**Data:** NASA C-MAPSS turbofan degradation simulation (Saxena & Goebel, 2008), used as a proxy for the gas-generator core of the fictitious aeroderivative units GT-101/GT-102. Download it first with `python scripts/download_data.py`.

In [ ]:
import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
from sklearn.ensemble import RandomForestClassifier
from sklearn.metrics import average_precision_score
from sklearn.model_selection import GroupShuffleSplit, train_test_split

from industrial_ai.ml.data import (
    SENSOR_COLUMNS,
    SENSOR_DESCRIPTIONS,
    SETTING_COLUMNS,
    load_rul,
    load_trajectories,
)
from industrial_ai.ml.labels import LABEL_COLUMN, add_failure_label, add_rul_test, add_rul_train

pd.set_option("display.max_columns", 40)
SUBSET = "FD001"
HORIZON = 30

## 1. Load and validate

The loaders raise an error if a file does not have the expected structure (26 columns, no missing values, contiguous cycles starting at 1 in every unit). Getting here without errors is already a data-quality check.

In [ ]:
train = add_rul_train(load_trajectories(SUBSET, "train"))
test = add_rul_test(load_trajectories(SUBSET, "test"), load_rul(SUBSET))

print(f"train: {len(train):,} rows · {train['unit'].nunique()} units")
print(f"test:  {len(test):,} rows · {test['unit'].nunique()} units")
train.head()

## 2. Lifetimes of the training units

Every training unit runs until failure, so its number of cycles is its lifetime.
**Think about:** how much do lifetimes vary? Why does that matter when we use a fixed horizon of 30 cycles?

In [ ]:
lifetimes = train.groupby("unit")["cycle"].max()

fig, ax = plt.subplots(figsize=(8, 3))
ax.hist(lifetimes, bins=20)
ax.set(title="Training units: cycles until failure", xlabel="cycles", ylabel="units")
plt.show()

lifetimes.describe()

## 3. Data quality: missing values and uninformative columns

A column that barely changes cannot carry information about degradation. We look at the number of distinct values and the standard deviation of every setting and sensor.
**Write down** which columns look constant or near-constant; the decision to drop them is made in block 1.2 and recorded in `docs/ml.md`.

In [ ]:
measurement_columns = SETTING_COLUMNS + SENSOR_COLUMNS
quality = pd.DataFrame(
    {
        "missing": train[measurement_columns].isna().sum(),
        "n_unique": train[measurement_columns].nunique(),
        "std": train[measurement_columns].std(),
    }
)
quality["description"] = quality.index.map(SENSOR_DESCRIPTIONS)
quality.sort_values("std")

## 4. Operating conditions

FD001 is documented as a single operating condition, so the three settings should only show small variations. If they varied a lot, sensors would have to be normalised per operating regime (that is the case in FD002 and FD004).

In [ ]:
train[SETTING_COLUMNS].describe().T

## 5. Degradation signal: sensors aligned by remaining useful life

Plotting against cycle mixes units with very different lifetimes. Aligning every unit by its **RUL** (failure on the right) shows whether a sensor drifts as failure approaches.

In [ ]:
candidates = [column for column in SENSOR_COLUMNS if train[column].nunique() > 1]
sample_units = train["unit"].drop_duplicates().sample(10, random_state=42)

n_cols = 4
n_rows = int(np.ceil(len(candidates) / n_cols))
fig, axes = plt.subplots(n_rows, n_cols, figsize=(16, 2.6 * n_rows), sharex=True)
for ax, sensor in zip(axes.flat, candidates, strict=False):
    for unit in sample_units:
        unit_data = train[train["unit"] == unit]
        ax.plot(unit_data["rul"], unit_data[sensor], linewidth=0.7, alpha=0.7)
    short_name = SENSOR_DESCRIPTIONS[sensor].split(" - ")[0]
    ax.set_title(f"{sensor} · {short_name}", fontsize=9)
for ax in axes.flat[len(candidates) :]:
    ax.set_visible(False)
axes.flat[0].invert_xaxis()  # shared x axis: inverting once flips every panel
fig.supxlabel("RUL (cycles left) — failure at the right edge")
fig.tight_layout()
plt.show()

## 6. Monotonic association with RUL

Spearman correlation measures whether a sensor tends to go up (or down) as RUL decreases, without assuming a linear relationship.

Two cautions for the interview:
- **Correlation is not causation.** A sensor can correlate with RUL because it reacts to the same underlying degradation, not because it causes it.
- These correlations pool all units, so they mix *within-unit* degradation with *between-unit* differences.

In [ ]:
correlation_with_rul = (
    train[candidates + ["rul"]]
    .corr(method="spearman")["rul"]
    .drop("rul")
    .sort_values(key=np.abs, ascending=False)
)
correlation_with_rul.to_frame("spearman_vs_rul").assign(
    description=lambda df: df.index.map(SENSOR_DESCRIPTIONS)
)

## 7. Label balance

Positive label = the unit fails within the next `HORIZON` cycles (RUL ≤ horizon). The horizon is a business choice: how much warning does maintenance need to plan an intervention? We compare a few values to see how it changes the class balance.

In [ ]:
balance = pd.DataFrame(
    {
        horizon: add_failure_label(train, horizon)[LABEL_COLUMN].value_counts(normalize=True)
        for horizon in (15, 30, 50)
    }
).T.rename(columns={0: "negative", 1: "positive"})
balance.index.name = "horizon"
balance.round(3)

## 8. Test set: truncated trajectories

Test units stop **before** failure. The RUL file gives the true remaining life after the last observed cycle, from which the RUL of every earlier cycle is reconstructed. Because trajectories are truncated, the positive rate in the test set is expected to differ from training.

In [ ]:
test_summary = test.groupby("unit").agg(
    observed_cycles=("cycle", "max"), rul_at_last_cycle=("rul", "min")
)

fig, ax = plt.subplots(figsize=(8, 3))
ax.hist(test_summary["rul_at_last_cycle"], bins=20)
ax.axvline(HORIZON, color="black", linestyle="--", label=f"horizon = {HORIZON}")
ax.set(title="Test units: true RUL after the last observed cycle", xlabel="cycles")
ax.legend()
plt.show()

test_labelled = add_failure_label(test, HORIZON)
print(f"positive rate in test (all cycles): {test_labelled[LABEL_COLUMN].mean():.3f}")
test_summary.describe()

## 9. Leakage risk: random rows vs whole units

Consecutive cycles of the same engine are very similar and share that engine's own baseline. With a **random row split**, the model is evaluated on rows whose neighbours it saw during training. With a **split by unit**, every test engine is completely new, which is the real situation.

Experiment: same model, same raw sensors, two ways of splitting. One split is noisy, so read the gap as an indication; block 1.2 uses grouped cross-validation. If the gap is small here, it usually grows once we add rolling features, because overlapping windows make neighbouring rows share even more information.

In [ ]:
labelled = add_failure_label(train, HORIZON)
X = labelled[candidates]
y = labelled[LABEL_COLUMN]
groups = labelled["unit"]


def pr_auc_for_split(train_idx, test_idx):
    model = RandomForestClassifier(n_estimators=200, min_samples_leaf=5, n_jobs=-1, random_state=42)
    model.fit(X.iloc[train_idx], y.iloc[train_idx])
    scores = model.predict_proba(X.iloc[test_idx])[:, 1]
    return average_precision_score(y.iloc[test_idx], scores)


rows = np.arange(len(labelled))
random_train, random_test = train_test_split(rows, test_size=0.2, random_state=42, stratify=y)
splitter = GroupShuffleSplit(n_splits=1, test_size=0.2, random_state=42)
group_train, group_test = next(splitter.split(X, y, groups))

print(f"random row split  PR-AUC: {pr_auc_for_split(random_train, random_test):.3f}")
print(f"split by unit     PR-AUC: {pr_auc_for_split(group_train, group_test):.3f}")
print(f"positive rate (baseline PR-AUC of a random model): {y.mean():.3f}")

## 10. Findings

Fill this in after running the notebook. These notes feed the decisions of block 1.2 and `docs/ml.md`.

- **Size:** rows, units, lifetime range (min / median / max):
- **Quality:** missing values, constant or near-constant columns to drop:
- **Operating conditions:** do the settings confirm a single regime?
- **Degradation signal:** sensors with the clearest trend as RUL decreases:
- **Label balance:** positive rate at horizon 30 in train and in test:
- **Leakage experiment:** PR-AUC with random split vs split by unit, and your interpretation:
- **Open questions:**